# Thống kê có trọng số: tính từ bảng tần suất và tỉ số, so với pandas

Hai tình huống DA gặp hằng tuần:

1. **Chỉ có bảng đã gộp.** Team BI của nhà mạng không cho xuất dữ liệu từng khách (dữ liệu cá nhân), chỉ gửi
   bảng tần suất `tenure` (số tháng gắn bó) → số khách. Finance cần mean, SD, median, p25, p75 của `tenure` từ
   bảng đó. Bộ `telco-churn` (7 043 khách, IBM) cho phép bạn kiểm lại đáp án trên dữ liệu gốc.
2. **Tỉ số ở mỗi dòng.** Bộ `store-sales` (Rossmann, 1 115 cửa hàng, 2013-2015) có doanh số và số khách mỗi
   cửa hàng-ngày. "Doanh số mỗi khách" là tổng/tổng hay trung bình các tỉ số? Trọng số quyết định.

Việc của bạn: tự viết mean, SD và quantile có trọng số, chứng minh chúng khớp pandas/numpy khi "nhân bản" mỗi
dòng theo trọng số, rồi **làm phần trắc nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và in kết quả; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/07-descriptive-eda/exercises/basic/02-weighted-stats -q`

In [ ]:
# explore: tìm gốc repo, tải hai bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

telco = pd.read_csv(ensure("telco-churn") / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
train = pd.read_csv(ensure("store-sales") / "train.csv", dtype={"StateHoliday": str}, parse_dates=["Date"])
# bảng tần suất mà team BI gửi: mỗi dòng là một giá trị tenure và số khách có giá trị đó
tenure_freq = telco["tenure"].value_counts().sort_index().rename_axis("tenure").reset_index(name="customers")
print(len(telco), "khách;", len(tenure_freq), "giá trị tenure khác nhau")
tenure_freq.head()

In [ ]:
import numpy as np
import pandas as pd

## 1. Mean có trọng số

$$
\bar x_w = \frac{\sum_i w_i x_i}{\sum_i w_i}
$$

`weighted_mean(x, w) -> float`. `x`, `w` là dãy cùng độ dài (list, `np.ndarray` hoặc `pd.Series`). Quy tắc:

- bỏ những cặp có `x` hoặc `w` là NaN;
- `ValueError` nếu hai dãy khác độ dài, có trọng số âm, hoặc tổng trọng số (sau khi bỏ NaN) bằng 0.

Với `w` là số khách của mỗi giá trị `tenure`, kết quả phải bằng `telco.tenure.mean()`.

In [ ]:
def weighted_mean(x, w) -> float:
    """Mean có trọng số; bỏ cặp NaN; ValueError nếu khác độ dài, w < 0 hoặc tổng w = 0."""
    # TODO: x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float); kiểm độ dài
    # TODO: bỏ cặp NaN, kiểm w >= 0 và tổng > 0
    # TODO: sum(w * x) / sum(w)
    raise NotImplementedError

In [ ]:
# explore: mean từ bảng tần suất so với dữ liệu gốc; và cái bẫy "trung bình các giá trị khác nhau"
print("từ bảng tần suất:", weighted_mean(tenure_freq.tenure, tenure_freq.customers))
print("trên dữ liệu gốc: ", telco.tenure.mean())
print("mean của", len(tenure_freq), "giá trị tenure (sai, mỗi giá trị một phiếu):", tenure_freq.tenure.mean())

# tỉ lệ churn theo hợp đồng: trung bình các tỉ lệ hay có trọng số?
by_contract = telco.assign(churned=telco.Churn.eq("Yes")).groupby("Contract").churned.agg(rate="mean", customers="size")
print(by_contract)
print(f"mean ba tỉ lệ {by_contract.rate.mean():.1%}; có trọng số {weighted_mean(by_contract.rate, by_contract.customers):.1%}")

## 2. Độ lệch chuẩn có trọng số tần suất

Trọng số ở đây là **frequency weights**: `w = 5` nghĩa là giá trị đó xuất hiện 5 lần. Khi đó n là tổng trọng số,
và SD mẫu chia cho tổng trọng số trừ `ddof`:

$$
s_w = \sqrt{\frac{\sum_i w_i (x_i - \bar x_w)^2}{\sum_i w_i - \text{ddof}}}
$$

`weighted_std(x, w, ddof=1) -> float`, cùng quy tắc làm sạch và kiểm lỗi như `weighted_mean`. Với trọng số
nguyên, `ddof=1` phải khớp `pd.Series(np.repeat(x, w)).std()` và `ddof=0` khớp `np.std(np.repeat(x, w))`.

In [ ]:
def weighted_std(x, w, ddof: int = 1) -> float:
    """SD có trọng số tần suất: sqrt(sum w (x - mean_w)^2 / (sum w - ddof))."""
    # TODO: làm sạch và kiểm lỗi như weighted_mean (nên tách một hàm phụ dùng chung)
    # TODO: mean có trọng số, rồi tổng bình phương độ lệch có trọng số
    raise NotImplementedError

In [ ]:
# explore: SD mẫu và SD tổng thể từ bảng tần suất
print("SD mẫu:     ", weighted_std(tenure_freq.tenure, tenure_freq.customers), "| pandas:", telco.tenure.std())
print("SD tổng thể:", weighted_std(tenure_freq.tenure, tenure_freq.customers, ddof=0), "| numpy:", np.std(telco.tenure))

## 3. Quantile có trọng số

Median của bảng tần suất là giá trị mà ở đó **số khách cộng dồn** vượt nửa tổng số khách. Tổng quát cho mức `q`:
sắp `x` tăng dần (trọng số đi theo), cộng dồn trọng số, trả về **giá trị `x` đầu tiên có trọng số cộng dồn
`>= q * tổng trọng số`**. Đây là định nghĩa "inverted CDF": không nội suy, kết quả luôn là một giá trị có thật
trong dữ liệu, khớp `np.quantile(np.repeat(x, w), q, method="inverted_cdf")` khi trọng số nguyên.

`weighted_quantile(x, w, q) -> float`: làm sạch như trên, bỏ thêm các cặp có `w == 0`; `ValueError` nếu `q`
ngoài `[0, 1]`.

Lưu ý: pandas `quantile` mặc định **nội suy tuyến tính**, nên với số quan sát chẵn hoặc `q` lẻ kết quả có thể
khác một chút. Với `tenure` (7 043 khách, giá trị nguyên) median và các tứ phân vị trùng nhau.

In [ ]:
def weighted_quantile(x, w, q: float) -> float:
    """Giá trị x đầu tiên (theo thứ tự tăng) có trọng số cộng dồn >= q * tổng trọng số."""
    # TODO: kiểm 0 <= q <= 1; làm sạch, bỏ thêm cặp w == 0
    # TODO: sắp theo x (np.argsort), np.cumsum(w), np.searchsorted(..., side="left")
    raise NotImplementedError

In [ ]:
# explore: median và tứ phân vị từ bảng tần suất
for q in (0.25, 0.5, 0.75):
    print(q, weighted_quantile(tenure_freq.tenure, tenure_freq.customers, q), "| pandas:", telco.tenure.quantile(q))

## 4. Doanh số mỗi khách của Rossmann: trọng số nào?

Mỗi dòng ngày mở cửa có `Sales` và `Customers`, nên có tỉ số `Sales / Customers` (EUR mỗi khách của cửa hàng
đó trong ngày đó). Cell dưới so ba con số:

- mean không trọng số của các tỉ số (mỗi cửa hàng-ngày một phiếu);
- mean các tỉ số **có trọng số theo số khách** (mỗi khách một phiếu);
- tổng doanh số / tổng số khách.

Hai con số sau luôn bằng nhau (vì sao?). Con số nào trả lời "mỗi khách mang lại bao nhiêu tiền cho chuỗi"?
Median có trọng số theo khách nói gì khác median không trọng số?

In [ ]:
# explore: ba cách tính "doanh số mỗi khách"
open_days = train[(train.Open == 1) & (train.Sales > 0)]
ratio = open_days.Sales / open_days.Customers
print(f"{len(open_days):,} cửa hàng-ngày mở cửa")
print("mean các tỉ số:               ", round(ratio.mean(), 4))
print("mean có trọng số theo số khách:", round(weighted_mean(ratio, open_days.Customers), 4))
print("tổng / tổng:                   ", round(open_days.Sales.sum() / open_days.Customers.sum(), 4))
print("median không trọng số:", ratio.median(), "| median theo số khách:", round(weighted_quantile(ratio, open_days.Customers, 0.5), 4))

## 5. Trắc nghiệm

Xong thì mở app, làm phần **trắc nghiệm** của bài. Các câu hỏi dùng đúng những con số các cell trên in ra.